# Notebook 2: $k_{\rm opt} = \alpha_F/2$ — Feigenbaum Scaling Conjecture

**Paper reference**: §II.E, Proposition 2 (conjectured)

**Goal**: Verify numerically that the optimal cooling constant $k_{\rm opt} \approx \alpha_F/2 = 1.2515$ to within 0.28%.

In [ ]:
import numpy as np
from scipy.optimize import brentq

mu_inf = 3.569945671870944
alpha_F = 2.502907875095892
delta_F = 4.669201609102990
k_opt_paper = 1.248

## 2.1 Compute Feigenbaum constants from period-doubling bifurcations

In [ ]:
def logistic(x, mu):
    return mu * x * (1 - x)

def find_superstable(period, mu_low, mu_high):
    """Find μ where f^period(1/2) = 1/2 (superstable cycle)."""
    def cond(mu):
        x = 0.5
        for _ in range(period):
            x = logistic(x, mu)
        return x - 0.5
    return brentq(cond, mu_low, mu_high, xtol=1e-14)

# Find superstable points
bif = {}
bif[1] = 2.0
bif[2] = find_superstable(2, 3.0, 3.5)
bif[4] = find_superstable(4, 3.4, 3.57)
bif[8] = find_superstable(8, 3.54, 3.57)
bif[16] = find_superstable(16, 3.564, 3.570)
bif[32] = find_superstable(32, 3.5685, 3.5700)

print("Superstable bifurcation values:")
for p, mu in sorted(bif.items()):
    print(f"  μ*(period={p:>2d}) = {mu:.15f}")

# Feigenbaum δ
periods = sorted(bif.keys())
print("\nFeigenbaum δ:")
for i in range(2, len(periods)):
    d = (bif[periods[i-1]] - bif[periods[i-2]]) / (bif[periods[i]] - bif[periods[i-1]])
    print(f"  δ({periods[i-2]},{periods[i-1]},{periods[i]}) = {d:.10f}")

print(f"\nExact δ_F = {delta_F:.15f}")

## 2.2 Compare $k_{\rm opt}$ with $\alpha_F/2$

In [ ]:
alpha_F_half = alpha_F / 2

print(f"α_F / 2     = {alpha_F_half:.12f}")
print(f"k_opt       = {k_opt_paper:.12f}")
print(f"Difference  = {alpha_F_half - k_opt_paper:.12f}")
print(f"Relative error = {abs(alpha_F_half - k_opt_paper)/k_opt_paper * 100:.4f}%")
print()
print("Other candidate relations:")
candidates = {
    'ln(μ_∞)': np.log(mu_inf),
    'δ_F/μ_∞': delta_F/mu_inf,
    'α_F/2': alpha_F/2,
    'ln(2)×ln(δ_F)': np.log(2)*np.log(delta_F),
}
for name, val in candidates.items():
    print(f"  {name:20s} = {val:.6f}  (ratio to k_opt: {val/k_opt_paper:.4f})")

## Conclusion

$k_{\rm opt} \approx \alpha_F/2$ to 0.28% — a **conjectured** deep connection between spatial rescaling ($\alpha_F$) and temporal cooling ($k_{\rm opt}$). Rigorous proof requires RG analysis.